In [1]:
import pandas as pd
import numpy as np

In [10]:
df=pd.read_csv(r"../../data/processed/merged_ieee.csv")
df.tail()

,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
590535,3577535,0,15811047,49.00,W,6550,NaN,150.0,visa,226.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590536,3577536,0,15811049,39.50,W,10444,225.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590537,3577537,0,15811079,30.95,W,12037,595.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590538,3577538,0,15811088,117.00,W,7826,481.0,150.0,mastercard,224.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
590539,3577539,0,15811131,279.95,W,15066,170.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
df=df.sort_values('TransactionDT').reset_index(drop=True)

In [ ]:
n=len(df)
train_end=int(0.8*n)
val_end=int(0.9*n)
df_train=df.iloc[:train_end].copy()
df_val=df.iloc[train_end:val_end].copy()
df_test=df.iloc[val_end:].copy()

In [13]:
print(f"Train: {len(df_train)} rows")
print(f"Fraud rate: {df_train['isFraud'].mean()}")
print(f"Validation: {len(df_val)} rows")
print(f"Fraud rate: {df_val['isFraud'].mean()}")
print(f"Train: {len(df_test)} rows")
print(f"Fraud rate: {df_test['isFraud'].mean()}")


Train: 472432 rows
Fraud rate: 0.03513521522674162
Validation: 59054 rows
Fraud rate: 0.03134419345006265
Train: 59054 rows
Fraud rate: 0.03747417617773563


In [14]:
#Save the splits to csvs

df_train.to_csv("../../data/processed/train.csv", index=False)
df_val.to_csv("../../data/processed/val.csv", index=False)
df_test.to_csv("../../data/processed/test.csv", index=False)

In [10]:
df[['card1','card2','card3','card4','card5']]

,card1,card2,card3,card4,card5
0,13926,NaN,150.0,discover,142.0
1,2755,404.0,150.0,mastercard,102.0
2,4663,490.0,150.0,visa,166.0
3,18132,567.0,150.0,mastercard,117.0
4,4497,514.0,150.0,mastercard,102.0
...,...,...,...,...,...
590535,6550,NaN,150.0,visa,226.0
590536,10444,225.0,150.0,mastercard,224.0
590537,12037,595.0,150.0,mastercard,224.0
590538,7826,481.0,150.0,mastercard,224.0


In [11]:
df[['card1','card2','card3','card4','card5']].nunique()

card1    13553
card2      500
card3      114
card4        4
card5      119
dtype: int64

In [16]:
df_train=pd.read_csv(r"../../data/processed/train.csv")
df_val=pd.read_csv(r"../../data/processed/val.csv")
df_test=pd.read_csv(r"../../data/processed/test.csv")

In [17]:
def add_time_features(df):
    df=df.copy()
    df['hour']=df['TransactionDT']%86400//3600
    df['day_of_week']=(df['TransactionDT']//86400)%7
    df['is_night']= df['hour'].between(0, 5).astype(int)
    return df

def add_amount_features(df,df_train=None):
    df=df.copy()
    df['log_amount']=np.log1p(df['TransactionAmt'])
    if df_train is None:
        card_mean=df.groupby('card1')['TransactionAmt'].mean().rename('card1_mean')
    else:
        card_mean=df_train.groupby('card1')['TransactionAmt'].mean().rename('card1_mean')
    df=df.merge(card_mean,on='card1',how='left')
    df['card1_mean']=df['card1_mean'].fillna(df['TransactionAmt'].median())
    df['amount_to_card_mean']=df['TransactionAmt']/(df['card1_mean']+1e-8)
    df.drop(columns='card1_mean',inplace=True)
    return df

df_train = add_time_features(df_train)
df_train = add_amount_features(df_train, df_train=None)

df_val  = add_time_features(df_val)
df_val  = add_amount_features(df_val, df_train=df_train)

df_test = add_time_features(df_test)
df_test = add_amount_features(df_test, df_train=df_train)

In [18]:
def add_email_features(df,df_train=None):
    df=df.copy()
    df['email_domain_mismatch']=(df['P_emaildomain'].fillna('unknown')!=df['R_emaildomain'].fillna('unknown')).astype(int)
    high_risk_domains=['gmail.com','yahoo.com','hotmail.com','anonymous.com']
    df['purchaser_email_risk']=df['P_emaildomain'].isin(high_risk_domains).astype(int)
    source=df_train if df_train is not None else df
    p_freq=source['P_emaildomain'].value_counts(normalize=True)
    df['p_email_freq']=df['P_emaildomain'].map(p_freq).fillna(0)
    return df

def add_device_features(df):
    df=df.copy()
    df['is_mobile']=(df['DeviceType']=='mobile').astype(int)
    return df

def add_card_features(df,df_train='None'):
    df=df.copy()
    source=df_train if df_train is not None else df
    
    for col in ['card4','card6']:
        freq=source[col].value_counts(normalize=True)
        df[f'{col}_freq']=df[col].map(freq).fillna(0)
    return df

df_train = add_email_features(df_train, df_train=None)
df_train = add_device_features(df_train)
df_train = add_card_features(df_train, df_train=None)

df_val = add_email_features(df_val, df_train=df_train)
df_val = add_device_features(df_val)
df_val = add_card_features(df_val, df_train=df_train)

df_test = add_email_features(df_test, df_train=df_train)
df_test = add_device_features(df_test)
df_test = add_card_features(df_test, df_train=df_train)


In [19]:
def add_product_features(df_train,df_val,df_test):
    prod_train=pd.get_dummies(df_train['ProductCD'],prefix='prod')
    prod_val=pd.get_dummies(df_val['ProductCD'],prefix='prod')
    prod_test=pd.get_dummies(df_test['ProductCD'],prefix='prod')
    for col in prod_train.columns:
        if col not in prod_val.columns:
            prod_val[col]=0
        if col not in prod_test.columns:
            prod_test[col]=0
    prod_val=prod_val[prod_train.columns]
    prod_test=prod_test[prod_train.columns]
    df_train=pd.concat([df_train,prod_train],axis=1)
    df_val=pd.concat([df_val,prod_val],axis=1)
    df_test=pd.concat([df_test,prod_test],axis=1)
    return df_train,df_val,df_test

df_train, df_val, df_test = add_product_features(df_train, df_val, df_test)

FEATURE_COLS = [
    # Time
    'hour', 'day_of_week', 'is_night',
    # Amount
    'log_amount', 'amount_to_card_mean',
    # Email
    'email_domain_mismatch', 'purchaser_email_risk', 'p_email_freq',
    # Device
    'is_mobile',
    # Card
    'card4_freq', 'card6_freq',
    # ProductCD one-hot (prod_W, prod_H, etc.)
] + [c for c in df_train.columns if c.startswith('prod_')]

In [20]:
X_train=df_train[FEATURE_COLS].fillna(0)
y_train=df_train['isFraud']

X_val=df_val[FEATURE_COLS].fillna(0)
y_val=df_val['isFraud']

X_test=df_test[FEATURE_COLS].fillna(0)
y_test=df_test['isFraud']

print('Feature matrix: ',X_train.shape)
print('Fraud case in training',y_train.sum())
print("Features: ",FEATURE_COLS)

Feature matrix:  (472432, 16)
Fraud case in training 16599
Features:  ['hour', 'day_of_week', 'is_night', 'log_amount', 'amount_to_card_mean', 'email_domain_mismatch', 'purchaser_email_risk', 'p_email_freq', 'is_mobile', 'card4_freq', 'card6_freq', 'prod_C', 'prod_H', 'prod_R', 'prod_S', 'prod_W']


In [21]:
df_train.to_csv("train_engineered.csv", index=False)
df_val.to_csv("val_engineered.csv", index=False)
df_test.to_csv("test_engineered.csv", index=False)